<h1 style=\"text-align: center; font-size: 50px;\">🎥 Advanced Recommender Systems with Tensorflow MLflow Integration</h1>

## Notebook Overview
- Start Execution
- User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Loading Data
- Memory-Based Collaborative Filtering
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 18.2 ms, sys: 2.83 ms, total: 21.1 ms
Wall time: 677 ms


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 4


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

## Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-15 02:59:24 - INFO - Notebook execution started.


## User Constants

In [3]:
MOVIE_ID = 5
RATING = 3.5

## Install and Import Libraries

In [4]:
# ------------------------ Data Manipulation ------------------------
import numpy as np
import pandas as pd

# # ------------------------ Statistical and Machine Learning tools ------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics import mean_squared_error
from math import sqrt
import scipy.sparse as sp
from scipy.sparse.linalg import svds

# ------------------------ Deep learning framework ------------------------
import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

# ------------------------ System Utilities ------------------------
import os
import warnings
import datetime
from pathlib import Path
import sys

# ------------------------ Visualization Libraries ------------------------
import matplotlib.pyplot as plt

# ------------------------ MLflow Integration ------------------------
import mlflow
from mlflow import MlflowClient
from mlflow.types.schema import Schema, ColSpec
from mlflow.models import ModelSignature

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.mlflow import Logger

from src.utils import (
    load_config,
)

from IPython import get_ipython

Note: you may need to restart the kernel to use updated packages.


2026-04-15 02:59:26.847917: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 02:59:26.861241: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776221966.877389    1840 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776221966.882098    1840 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 02:59:26.898297: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

CPU times: user 4.21 s, sys: 6.09 s, total: 10.3 s
Wall time: 6.16 s


## Configure Settings

In [5]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [6]:
# ------------------------- Paths -------------------------
DATA_PATH = "/home/jovyan/datafabric/tutorial/"
LOG_DIR = "/phoenix/tensorboard/tensorlogs/"
OUTPUT_DIR = "../model_artifacts"
ARTIFACT_PATH =  "movie_titles" 
# Name of the MLflow experiment for tracking performance and metrics
EXPERIMENT_NAME = "MovieRecommenderExperiment"
RUN_NAME = "Movie_Recommender_Run"
MODEL_NAME = "movie_titles"     

# Configuration paths
CONFIG_PATH = "../configs/config.yaml"
DEMO_FOLDER = "../demo"

# Load configuration
config = load_config(CONFIG_PATH)

logger.info("✅ Configuration loaded successfully")

2026-04-15 02:59:30 - INFO - ✅ Configuration loaded successfully


## Verify Assets

In [7]:
# Check whether the Dataset file exists
is_dataset_available = Path(DATA_PATH).exists()

# Log the configuration status of the dataset
if is_dataset_available:
    logger.info("The Dataset is properly configured.")
else:
    logger.info(
        "The Dataset is not properly configured. Please create and download the required assets "
        "in your project on AI Studio."
    )

2026-04-15 02:59:30 - INFO - The Dataset is properly configured.


## Loading Data

In [8]:
asset_folder = DATA_PATH

In [9]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv(f"{asset_folder}ml-100k/u.data", sep='\t', names=column_names)

In [10]:
movie_titles = pd.read_csv(f"{asset_folder}Movie_Id_Titles.csv")

In [11]:
df = pd.merge(df,movie_titles,on='item_id')

In [12]:
display(df.sample(2))
display(df.shape)

,user_id,item_id,rating,timestamp,title
58366,859,111,4,885776056,"Truth About Cats & Dogs, The (1996)"
66260,738,455,4,875350551,Jackie Chan's First Strike (1996)


(100000, 5)

In [13]:
train_data, test_data = train_test_split(df, test_size=0.25)

## Memory-Based Collaborative Filtering

In [14]:
n_users = df.user_id.nunique()
n_items = df.item_id.nunique()
#Create two user-item matrices, one for training and another for testing
train_data_matrix = np.zeros((n_users, n_items))
for line in train_data.itertuples():
    train_data_matrix[line[1]-1, line[2]-1] = line[3]  

test_data_matrix = np.zeros((n_users, n_items))
for line in test_data.itertuples():
    test_data_matrix[line[1]-1, line[2]-1] = line[3]

In [15]:
user_similarity = pairwise_distances(train_data_matrix, metric='cosine')
item_similarity = pairwise_distances(train_data_matrix.T, metric='cosine')

In [16]:
def predict(ratings, similarity, type='user'):
    """
    Predicts ratings using collaborative filtering based on user or item similarity.

    Parameters:
        ratings (array): A matrix where each row represents a user and each column represents an item.
        similarity (array): A similarity matrix representing relationships between users or items.
        type (str): Defines the type of prediction. Defaults to 'user'.

    Returns:
        array: A matrix of predicted ratings.
    """
    try:
        if type == 'user':
            mean_user_rating = ratings.mean(axis=1)
            #You use np.newaxis so that mean_user_rating has same format as ratings
            ratings_diff = (ratings - mean_user_rating[:, np.newaxis]) 
            pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
        elif type == 'item':
            pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])     
        return pred
    except Exception as e:
        logger.error(f"Error predicting ratings: {str(e)}")
        raise

In [17]:
item_prediction = predict(train_data_matrix, item_similarity, type='item')
user_prediction = predict(train_data_matrix, user_similarity, type='user')

### SVD

In [18]:
def rmse(prediction, ground_truth):
    """
    Computes the Root Mean Square Error (RMSE) between predicted values and ground truth values.

    Parameters:
        prediction (array-like): Predicted values.
        ground_truth (array-like): Actual values.

    Returns:
        float: The RMSE value.
    """
    try:
        prediction = prediction[ground_truth.nonzero()].flatten() 
        ground_truth = ground_truth[ground_truth.nonzero()].flatten()
        return sqrt(mean_squared_error(prediction, ground_truth))
    except Exception as e:
            logger.error(f"Error computing rmse: {str(e)}")
            raise

In [19]:
#get SVD components from train matrix. Choose k.
u, s, vt = svds(train_data_matrix, k = 20)
s_diag_matrix=np.diag(s)
X_pred = np.dot(np.dot(u, s_diag_matrix), vt)
logger.info('User-based CF MSE: ' + str(rmse(X_pred, test_data_matrix)))

2026-04-15 02:59:33 - INFO - User-based CF MSE: 2.71501550548084


## Logging Model to MLflow

In [20]:
def normalize_ratings(ratings, min_rating=1, max_rating=5):
    """Normalize ratings to 1-5 scale"""
    ratings = np.array(ratings)
    
    if ratings.max() == ratings.min():
        return np.full(ratings.shape, 3.0)  # Return middle value
    
    normalized = (ratings - ratings.min()) / (ratings.max() - ratings.min())
    scaled = normalized * (max_rating - min_rating) + min_rating
    
    return np.clip(scaled, min_rating, max_rating)

# MovieRecommender class removed - business logic moved to src/mlflow/model.py
# Using models-from-code approach with Logger for registration

output_dir = OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)
train_data_matrix_path = os.path.join(output_dir, "train_data_matrix.npy")
np.save(train_data_matrix_path, train_data_matrix)
movie_titles_path = os.path.join(output_dir, "movie_titles.csv")
movie_titles.to_csv(movie_titles_path, index=False)

In [21]:
logger.info(f'🚀 Starting the experiment: {EXPERIMENT_NAME}')

# Set the MLflow experiment name
mlflow.set_tracking_uri("/phoenix/mlflow")
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

# Start an MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:
    user_rmse = rmse(user_prediction, test_data_matrix)
    item_rmse = rmse(item_prediction, test_data_matrix)
    svd_rmse = rmse(X_pred, test_data_matrix)
    
    mlflow.log_metric("User_based_CF_RMSE", user_rmse)
    mlflow.log_metric("Item_based_CF_RMSE", item_rmse)
    mlflow.log_metric("User_based_CF_MSE_SVD", svd_rmse)
    # Print the artifact URI for reference
    logger.info(f"📁 Run's Artifact URI: {run.info.artifact_uri}")

    # Log the model to MLflow using new Logger pattern
    input_schema = Schema([
        ColSpec("long", "movie_id"),
        ColSpec("double", "rating")
    ])
    output_schema = Schema([
        ColSpec("string", "movie_title"),
        ColSpec("double", "prediction")
    ])
    signature = ModelSignature(inputs=input_schema, outputs=output_schema)
    
    Logger.log_model(
        train_data_matrix_path=train_data_matrix_path, 
        movie_titles_path=movie_titles_path, 
        config_path=CONFIG_PATH,
        demo_folder=DEMO_FOLDER,
        artifact_path=ARTIFACT_PATH,
        signature=signature
    )

    # Register the logged model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )

    logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

logger.info(f'✅ Registered the model: {MODEL_NAME}')

2026-04-15 02:59:33 - INFO - 🚀 Starting the experiment: MovieRecommenderExperiment
2026/04/15 02:59:33 INFO mlflow.tracking.fluent: Experiment with name 'MovieRecommenderExperiment' does not exist. Creating a new experiment.
2026-04-15 02:59:34 - INFO - 📁 Run's Artifact URI: /phoenix/mlflow/141668734921028528/3236a5f9af5e4792a6645a52ef159f33/artifacts
Successfully registered model 'movie_titles'.
2026/04/15 02:59:39 WARNING mlflow.tracking._model_registry.fluent: Run with id 3236a5f9af5e4792a6645a52ef159f33 has no artifacts at artifact path 'movie_titles', registering model based on models:/m-3dc7763e80a5420e9189eca00a47b0dd instead
Created version '1' of model 'movie_titles'.
2026-04-15 02:59:40 - INFO - ✅ Model registered successfully with run ID: 3236a5f9af5e4792a6645a52ef159f33
2026-04-15 02:59:40 - INFO - ✅ Registered the model: movie_titles


## Fetching the Latest Model Version from MLflow

In [22]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the model
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version  # Extract the latest model version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_model_version}")

# Print the latest model version and its signature
logger.info(f"Latest Model Version: {latest_model_version}")
logger.info(f"Model Signature: {model_info.signature}")

2026-04-15 02:59:41 - INFO - Latest Model Version: 1
2026-04-15 02:59:41 - INFO - Model Signature: inputs: 
  ['movie_id': long (required), 'rating': double (required)]
outputs: 
  ['movie_title': string (required), 'prediction': double (required)]
params: 
  None



## Loading the Model and Running Inference

In [23]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_model_version}")

df_input = pd.DataFrame({
   
    'movie_id': [310, 325, 340, 355, 370, 385, 400, 415, 430, 445],
    'rating': [3.0, 2.5, 4.0, 5.0, 1.0, 3.5, 4.5, 2.0, 5.0, 3.0],


})
prediction = model.predict(df_input)
logger.info(prediction)

2026-04-15 02:59:43 - INFO - [('Great Day in Harlem, A (1994)', 4.632374749101864), ('Rear Window (1954)', 4.389121004904259), ('Casablanca (1942)', 4.362442089551843), ('Star Kid (1997)', 4.35161694711028), ('Citizen Kane (1941)', 4.348195828283564)]


In [24]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-15 02:59:43 - INFO - ⏱️ Total execution time: 0m 19.15s


In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://zdocs.datascience.hp.com/docs/aistudio/overview).